# Chapter 45: an offline production-engineering example

Freshly run one normal constructed pipeline scenario, write its SQLite state and core artifacts, verify cash/NAV/P&L, then save one event-stage plot. These cells do not demonstrate the full crash, stop, recovery or idempotent-replay suite; those have separate documented engineering checks.

UTC event, availability, ingestion and as_of clocks must qualify no later than the signal; execution follows it. Units, freshness and finite numeric inputs are checked before fills. The accepted core binds run keys to fingerprints and publishes staged artifacts. Fixed configuration allows positive real-valued shares without certifying broker lot sizes; amount reconciliation tolerance is USD 10⁻⁶. Nonpositive momentum gives zero targets. No training, market adapter or deployment is added.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch45_production_quant_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import csv, json
import matplotlib.pyplot as plt
from quant_course.production_pipeline import default_config, constructed_inputs, run_pipeline


## 1. Default configuration and inputs

Inspect initial cash, currency, half-spread basis points, commission, security metadata and observation/signal times. These are explicit constructed inputs; future execution values are separate from decision observations.

In [ ]:
cfg = default_config()
inputs = constructed_inputs()

print("Initial Cash:", cfg["initial_cash"], cfg["currency"])
print("Half Spread (bps):", cfg["execution"]["half_spread_bps"])
print("Commission Rate:", cfg["execution"]["commission_rate"])
print("Securities metadata:", inputs["metadata"]["securities"])
print("Observations count:", len(inputs["observations"]))
print("Signal Time:", inputs["metadata"]["signal_time"])

## 2. One normal run

Call run_pipeline once with run_key=nb_demo_run, state.db and artifacts inside this fresh RUN_DIR. It checks inputs, computes momentum, applies pre-trade leverage/stress gates, persists fills, reconciles and publishes. The core supports additional recovery policies, but no injected failure, STOPPED case or replay is exercised by these cells. Plotting is an ancillary step after core publication, outside that transaction.

In [ ]:
run_dir = RUN_DIR
state_db, output_dir = run_dir / "state.db", run_dir / "artifacts"
summary = run_pipeline(config=cfg, inputs=inputs, run_key="nb_demo_run",
                       state_db=state_db, output_dir=output_dir)
print(json.dumps(summary, indent=2))
print("Preserved run:", run_dir.relative_to(ROOT))


## 3. Reconciliation

Check cash of USD 7,995.998, NAV of USD 9,995.998 and net P&L of −USD 4.002 within USD 10⁻⁶. Spread cost of USD 2.000 plus commission of USD 2.002 accounts for the cost; gross P&L is zero. PASS refers to those residual checks. The retained code’s “author candidate only” print is historical wording rather than current acceptance authority.

In [ ]:
assert abs(summary["final_cash"] - 7995.998) <= 1e-6
assert abs(summary["terminal_nav"] - 9995.998) <= 1e-6
assert abs(summary["net_pnl"] + 4.002) <= 1e-6
assert summary["reconciliation"]["status"] == "PASS"
assert summary["reconciliation"]["cash_reconciled"] is True
assert summary["reconciliation"]["positions_reconciled"] is True
assert summary["reconciliation"]["nav_pnl_reconciled"] is True
assert summary["gate_outcomes"]["balance_sheet_reconciliation"] == "PASS"
print("CONSTRUCTED baseline agrees with the hand calculation; author candidate only.")

## 4. Event stages within one period

Read ledger_events.csv and plot recorded cash/NAV by event stage. These are not multiple monthly research periods. The code includes an initial/final display fallback if no rows are found; the accepted core normally produces the CSV. Save one new PNG under RUN_DIR.

In [ ]:
ledger_csv_path = output_dir / "ledger_events.csv"
stages = []
cash_series = []
nav_series = []

if ledger_csv_path.exists():
    with open(ledger_csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            evt = row.get("event_type", "EVENT")
            ts = row.get("timestamp", "")
            short_time = ts[11:16] if len(ts) >= 16 else ts
            stages.append(f"{evt}\n({short_time})")
            cash_series.append(float(row.get("cash_after", 0.0)))
            nav_series.append(float(row.get("nav", 0.0)))

if not stages:
    stages = ["Initial", "Final"]
    cash_series = [summary["initial_cash"], summary["final_cash"]]
    nav_series = [summary["initial_cash"], summary["terminal_nav"]]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(stages, nav_series, marker="o", color="#2980b9", linewidth=2, label="NAV (USD)")
ax.plot(stages, cash_series, marker="s", color="#e67e22", linewidth=2, linestyle="--", label="Cash (USD)")
ax.set_ylabel("USD")
ax.set_title("CONSTRUCTED: Cash & NAV by recorded event", fontweight="bold")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="center right")
plt.tight_layout()
plt.savefig(RUN_DIR / "ch45_production_quant_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()